# Hands-On with Diffusion Models — Part 2
**IndabaX Uganda 2026 | Stable Diffusion & LoRA Style Fine-Tuning**

In this session we build directly on Part 1:

- We implemented DDPM **from scratch** — noise schedule, training loop, sampling, context mixing.
- Now we use **Hugging Face Diffusers** to work with Stable Diffusion XL (SDXL), a production model.
- We fine-tune it for **artistic style** using **LoRA** — the same concept as your context vector, scaled up enormously.
- The payoff: **style mixing** — the same weight slider you used for `female_weight`, but now blending art movements.

**Session arc** (mirrors Part 1, one level up):

| Part 1 | Part 2 |
|--------|--------|
| Understand the math | Understand what Diffusers gives you |
| Implement forward process | Run inference on a pretrained model |
| Implement training loop | Understand LoRA conceptually |
| Sample and visualise | Fine-tune and observe |
| Context mixing payoff | **Style mixing payoff** |

**Requirements:** GPU ≥16 GB VRAM · Python 3.10+ · Diffusers / PEFT / Datasets

In [ ]:
# Standard library
import os
import math
import random
from pathlib import Path

# Deep learning
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms

# Diffusers / HuggingFace
from diffusers import StableDiffusionXLPipeline, UNet2DConditionModel
from peft import LoraConfig

# Visualisation
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

# ── Part 2 modules (dataset.py, model.py, utils.py) ──────────────────────────
from dataset import WikiArtStyleDataset, load_wikiart
from model   import load_sdxl_components, configure_lora
from utils   import (
    load_config,
    encode_prompt,
    get_add_time_ids,
    train_epoch,
    plot_loss_curve,
    generate_styled,
    plot_style_sweep,
)

# ── Device setup ─────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16   # RTX 5090 (Blackwell) natively supports bfloat16

print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Load configuration from config.yaml ──────────────────────────────────────
config = load_config("config.yaml")

MODEL_ID          = config['model_id']
DATASET_SAVE_PATH = config['dataset_path']
STYLE_A_NAME      = config['style_a_name']
STYLE_B_NAME      = config['style_b_name']
IMAGE_SIZE        = config['image_size']
LORA_RANK         = config['lora_rank']
LORA_ALPHA        = config['lora_alpha']
TRAIN_EPOCHS      = config['num_epochs']
BATCH_SIZE        = config['batch_size']
GRAD_ACCUM        = config['gradient_accumulation_steps']
LEARNING_RATE     = config['learning_rate']
MAX_SAMPLES       = config['max_samples']
SAVE_DIR          = Path(config['save_dir'])
NUM_WORKERS       = config['num_workers']
SAVE_EVERY        = config['save_every_n_epochs']
N_INFERENCE_STEPS = config['num_inference_steps']
GUIDANCE_SCALE    = config['guidance_scale']
SEED              = config['seed']
NEGATIVE_PROMPT   = config['negative_prompt']
N_TEST_IMAGES     = config['n_test_images']
TEST_PROMPT       = config['test_prompt']
SWEEP_PROMPT      = config['sweep_prompt']

SAVE_DIR.mkdir(parents=True, exist_ok=True)
print("Configuration loaded ✓")
for k, v in config.items():
    print(f"  {k}: {v}")

---
## Part 0: Why Diffusers?

You spent Part 1 writing ~220 lines of carefully crafted Python to train and sample from a DDPM.
Now watch what Diffusers lets you do.

In [ ]:
# ── Load Stable Diffusion XL (downloads ~6.5 GB on first run) ────────────────
print("Loading SDXL pipeline …")
pipe_base = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    use_safetensors=True,
    variant="fp16",
).to(device)
pipe_base.set_progress_bar_config(disable=True)
print("✓ SDXL loaded")

# ── Generate a test image (no fine-tuning yet) ────────────────────────────────
generator = torch.Generator(device=device).manual_seed(SEED)

with torch.no_grad():
    result = pipe_base(
        prompt          = "a serene lake at sunrise, beautiful landscape painting",
        negative_prompt = NEGATIVE_PROMPT,
        num_inference_steps = N_INFERENCE_STEPS,
        guidance_scale  = GUIDANCE_SCALE,
        height=IMAGE_SIZE, width=IMAGE_SIZE,
        generator=generator,
    )

image_base = result.images[0]

plt.figure(figsize=(7, 7))
plt.imshow(image_base)
plt.axis("off")
plt.title(f"SDXL base — zero-shot, no fine-tuning ({IMAGE_SIZE} × {IMAGE_SIZE})", fontsize=11)
plt.tight_layout()
plt.show()
print(f"Image size: {image_base.size}")

---
## Part 1: WikiArt — The Training Dataset

### Why WikiArt?

In Part 1 we conditioned on **gender** (a binary label: female / male).
Now we scale up: **artistic style** across 27 movements — from Impressionism and Baroque to Ukiyo-e and Surrealism.

The WikiArt dataset on Hugging Face Hub (`huggan/wikiart`) is the ideal choice:
- **~80 000 paintings** from 129 artists, spanning five centuries of art history.
- Three rich label columns: `artist`, `style` (27 classes), `genre` (11 classes).
- Loads in one line of code — no manual download, no folder restructuring.

The style column is the direct pedagogical continuation of the gender context from Part 1:
it is a richer, more nuanced conditioning signal, one the model must *learn* rather than simply embed.

In [ ]:
# ── Load WikiArt (downloads ~24 GB on first run, then uses local cache) ───────
# load_wikiart() is defined in dataset.py.
# It checks for an existing local copy before hitting the network.
# DATASET_SAVE_PATH is loaded from config.yaml.

train_ds = load_wikiart(DATASET_SAVE_PATH)

In [ ]:
# ── Extract style metadata ────────────────────────────────────────────────────
style_feature = train_ds.features["style"]
style_names   = style_feature.names          # list of 27 style name strings
style_ids     = train_ds["style"]            # list of integer labels

# Count images per style
from collections import Counter
style_counts = Counter(style_ids)
sorted_items = sorted(style_counts.items(), key=lambda x: -x[1])

labels  = [style_names[i].replace("_", " ") for i, _ in sorted_items]
counts  = [c for _, c in sorted_items]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(labels, counts, color=plt.cm.tab20(np.linspace(0, 1, len(labels))))
ax.set_xlabel("Number of paintings")
ax.set_title("WikiArt: image count by artistic style (27 classes)", fontsize=13)
ax.invert_yaxis()

for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height() / 2,
            f"{count:,}", va="center", fontsize=8)

plt.tight_layout()
plt.show()

print("Full style list:")
for i, name in enumerate(style_names):
    print(f"  {i:2d}: {name.replace('_', ' ')}")

In [ ]:
# ── Visualise sample images from several styles ───────────────────────────────
PREVIEW_STYLES = [style_names[id] for id, count in sorted_items[:5]]
N_PER_STYLE    = 4

fig, axes = plt.subplots(len(PREVIEW_STYLES), N_PER_STYLE,
                          figsize=(N_PER_STYLE * 3, len(PREVIEW_STYLES) * 3))

for row, style_name in enumerate(PREVIEW_STYLES):
    style_id = style_names.index(style_name)
    # Collect indices for this style
    indices  = [i for i, s in enumerate(style_ids) if s == style_id]
    chosen   = random.sample(indices, min(N_PER_STYLE, len(indices)))

    for col, idx in enumerate(chosen):
        img = train_ds[idx]["image"].convert("RGB")
        img.thumbnail((256, 256))
        axes[row, col].imshow(img)
        axes[row, col].set_title(style_name)
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(style_name.replace("_", " "), fontsize=10, rotation=0,
                                       labelpad=90, va="center")

plt.suptitle("WikiArt sample images by style", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Look up style indices from the names loaded from config.yaml ──────────────
#
#   Impressionism  — soft light, visible brush strokes, vibrant colour
#   Ukiyo-e        — flat areas of colour, bold outlines, decorative composition
#
#   These two styles maximise visual contrast in the mixing payoff (Part 4).
#   STYLE_A_NAME and STYLE_B_NAME are already set from config.yaml.

STYLE_A_ID = style_names.index(STYLE_A_NAME)
STYLE_B_ID = style_names.index(STYLE_B_NAME)

count_a = style_counts[STYLE_A_ID]
count_b = style_counts[STYLE_B_ID]

print(f"Style A — {STYLE_A_NAME.replace('_', ' ')}: {count_a:,} paintings  (index {STYLE_A_ID})")
print(f"Style B — {STYLE_B_NAME.replace('_', ' ')}: {count_b:,} paintings  (index {STYLE_B_ID})")
print()
print("We will train a separate LoRA for each style, then blend them in Part 4.")
print("The blending weight is the direct analogue of `female_weight` from Part 1.")

---
## Part 2: Understanding LoRA

### What is LoRA and why does it matter?

Stable Diffusion XL has **860 million parameters**.
A full fine-tune would require:
- Storing a complete copy of all gradients (~3 × model size in memory).
- Saving a ~6 GB checkpoint per style.
- Hours of training even on high-end hardware.

**Low-Rank Adaptation (LoRA)** sidesteps this entirely.
Instead of updating every weight matrix $W$, we inject two tiny matrices $A$ and $B$ and only train those:

$$W' = W + \Delta W = W + BA$$

where $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times k}$, and $r \ll \min(d, k)$.

The rank $r$ is a hyperparameter (typically 4–64).
The base weights $W$ are **frozen** — they are never updated.

### The numbers speak for themselves

| | Full fine-tune | LoRA (rank 16) |
|---|---|---|
| Trainable params | 860 M | ~3 M |
| Checkpoint size | ~6.5 GB | ~25 MB |
| Training time | Hours | Minutes |
| Multiple styles | Separate 6 GB file per style | Stack adapters, blend with weights |

### Where does LoRA live in the architecture?

LoRA is injected into the **attention layers** of the U-Net:
specifically the query, key, value, and output projection matrices (`to_q`, `to_k`, `to_v`, `to_out`).

**Connection to Part 1:** In your custom `DDPMUnet`, the context conditioning is injected at the upsampling path:

```python
up2 = self.up1(cemb1 * up1 + temb1, down2)   # context embedding modulates features
up3 = self.up2(cemb2 * up2 + temb2, down1)
```

This is **cross-attention** — the context vector steers the feature maps.
In SDXL, cross-attention between text embeddings and image features is exactly what LoRA targets.
You already understand the concept.  LoRA just makes modifying those weights efficient.

### The analogy

> The base SDXL model is a vast artistic vocabulary.
> The LoRA is a **thin specialised lens** you place in front of it.
> The lens does not replace the vocabulary — it redirects it toward a style.

In [ ]:
# ── Visualise the parameter count difference ──────────────────────────────────
# We load only the UNet to count, then inject LoRA and count again.

print("Loading UNet for parameter analysis …")
unet_for_count = UNet2DConditionModel.from_pretrained(
    MODEL_ID, subfolder="unet", torch_dtype=DTYPE
)

total_params = sum(p.numel() for p in unet_for_count.parameters())
print(f"UNet total parameters   : {total_params:>15,}")

# Inject LoRA and count trainable parameters (rank and alpha from config)
lora_cfg_count = LoraConfig(
    r                 = LORA_RANK,
    lora_alpha        = LORA_ALPHA,
    init_lora_weights = "gaussian",
    target_modules    = config['lora_target_modules'],
)
unet_for_count.add_adapter(lora_cfg_count)

lora_params = sum(p.numel() for n, p in unet_for_count.named_parameters() if "lora" in n.lower())
print(f"LoRA trainable params   : {lora_params:>15,}")
print(f"Reduction               : {100 * lora_params / total_params:.2f}% of the full model")

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
categories = ["Full fine-tune (all UNet params)", f"LoRA (rank {LORA_RANK}) (only adapter params)"]
values     = [total_params / 1e6, lora_params / 1e6]
colors     = ["#e74c3c", "#2ecc71"]
bars       = ax.bar(categories, values, color=colors, width=0.4)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f"{val:.1f} M", ha="center", va="bottom", fontsize=12, fontweight="bold")

ax.set_ylabel("Parameters (millions)")
ax.set_title("Trainable parameters: full fine-tune vs LoRA", fontsize=12)
ax.set_yscale("log")
plt.tight_layout()
plt.show()

# Free memory
del unet_for_count
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("\nMemory cleared.")

---
## Part 3a: Impressionism LoRA Fine-Tuning

### The plan

We train the **first LoRA** on Impressionism paintings from WikiArt.

**Training recipe:**
1. Filter WikiArt to Impressionism images.
2. Generate captions: `"a painting in the style of impressionism"`.
3. Encode images → latent space (VAE).
4. Encode captions → text embeddings (two CLIP encoders).
5. Add noise at a random timestep.
6. Forward pass through the frozen UNet + active LoRA.
7. MSE loss between predicted noise and actual noise.
8. Backprop through LoRA matrices only.
9. Save adapter weights — periodic checkpoints **and** a best-loss checkpoint.

After training we immediately test the Impressionism LoRA before moving on to Ukiyo-e.

> **Live demo tip:** Start training running now, walk through the theory above, then come back to see the first checkpoint.

In [ ]:
# WikiArtStyleDataset is defined in dataset.py.
# It filters WikiArt to a single style and returns (image_tensor, caption) pairs.
# image_size is loaded from config.yaml.

print("Building dataset objects …")
ds_impressionism = WikiArtStyleDataset(train_ds, STYLE_A_ID, STYLE_A_NAME, image_size=IMAGE_SIZE)
ds_ukiyo_e       = WikiArtStyleDataset(train_ds, STYLE_B_ID, STYLE_B_NAME, image_size=IMAGE_SIZE)

sample = ds_impressionism[0]
print(f"  Image tensor shape : {sample['image'].shape}")   # [3, IMAGE_SIZE, IMAGE_SIZE]
print(f"  Caption            : {sample['caption']}")

In [ ]:
# ── Load all SDXL sub-models for training (defined in model.py) ───────────────
# Text encoders and VAE are frozen inside load_sdxl_components().
# Only the UNet will receive LoRA adapters in the next cell.
# MODEL_ID is loaded from config.yaml.

noise_scheduler, tokenizer_1, tokenizer_2, text_encoder_1, text_encoder_2, vae, unet = \
    load_sdxl_components(MODEL_ID, DTYPE, device)

In [ ]:
# ── Inject LoRA adapters into the UNet (defined in model.py) ──────────────────
# configure_lora() adds low-rank matrices to the attention layers, enables
# gradient checkpointing, and freezes all non-LoRA parameters.
# LORA_RANK, LORA_ALPHA, and target modules are all loaded from config.yaml.

unet, lora_config = configure_lora(
    unet,
    rank           = LORA_RANK,
    alpha          = LORA_ALPHA,
    target_modules = config['lora_target_modules'],
)

### Training helpers (from `utils.py`)

The functions used inside the training loop are defined in `utils.py`:

| Function | Purpose |
|----------|---------|
| `encode_prompt(captions, ...)` | Tokenise and embed text with both SDXL CLIP encoders |
| `get_add_time_ids(batch_size, ...)` | Build SDXL's image-size conditioning tensor |
| `train_epoch(unet, dataloader, ...)` | One epoch: VAE encode → add noise → UNet forward → MSE loss → backprop |

Open `utils.py` to see the full implementations with step-by-step comments.

In [ ]:
SKIP_TRAINING_A = False   # ← Set True to skip Impressionism training

# ─────────────────────────────────────────────────────────────────────────────
if not SKIP_TRAINING_A:
    print(f"Training {STYLE_A_NAME} LoRA for {TRAIN_EPOCHS} epoch(s) …")
    print(f"  Batch size    : {BATCH_SIZE}  (grad accum steps: {GRAD_ACCUM})")
    print(f"  Learning rate : {LEARNING_RATE}")
    print(f"  Max samples   : {MAX_SAMPLES}")

    loader_a = DataLoader(
        WikiArtStyleDataset(train_ds, STYLE_A_ID, STYLE_A_NAME,
                            image_size=IMAGE_SIZE, max_samples=MAX_SAMPLES),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
    )

    optimizer_a = torch.optim.AdamW(
        [p for p in unet.parameters() if p.requires_grad],
        lr=LEARNING_RATE, weight_decay=1e-2,
    )

    save_dir_a = SAVE_DIR / STYLE_A_NAME.lower()
    save_dir_a.mkdir(parents=True, exist_ok=True)

    loss_history_a = []
    best_loss_a    = float("inf")

    for epoch in range(1, TRAIN_EPOCHS + 1):
        avg_loss = train_epoch(
            unet, loader_a, optimizer_a, noise_scheduler, vae,
            tokenizer_1, tokenizer_2, text_encoder_1, text_encoder_2,
            device, DTYPE, epoch, STYLE_A_NAME, image_size=IMAGE_SIZE,
        )
        loss_history_a.append(avg_loss)
        print(f"  Epoch {epoch}/{TRAIN_EPOCHS}  loss = {avg_loss:.4f}")

        # Periodic checkpoint
        if epoch % SAVE_EVERY == 0 or epoch == TRAIN_EPOCHS:
            ckpt_path = save_dir_a / f"epoch_{epoch}"
            unet.save_pretrained(str(ckpt_path))
            print(f"    → Checkpoint saved: epoch_{epoch}/")

        # Best checkpoint
        if avg_loss < best_loss_a:
            best_loss_a = avg_loss
            unet.save_pretrained(str(save_dir_a / "best"))
            print(f"    ✓ New best! Loss: {best_loss_a:.4f}  → best/")

    # Final save
    unet.save_pretrained(str(save_dir_a / "final"))
    size_mb = sum(f.stat().st_size for f in (save_dir_a / "final").glob("*")) / 1e6
    print(f"\n✓ {STYLE_A_NAME} LoRA complete — best loss: {best_loss_a:.4f}")
    print(f"  Saved at : {save_dir_a}/  (best/, final/, epoch_N/)")
    print(f"  Size     : {size_mb:.1f} MB (final checkpoint)")

    plot_loss_curve(loss_history_a, STYLE_A_NAME, color="steelblue")
else:
    print(f"SKIP_TRAINING_A = True — skipping {STYLE_A_NAME} training.")
    print(f"  Expected weights at: {SAVE_DIR / STYLE_A_NAME.lower()}/")

### Testing the Impressionism LoRA

Before moving on, we verify that the Impressionism LoRA has captured the style correctly.
We load the **best saved checkpoint** (lowest training loss) and run inference with the test prompt from `config.yaml`.

The images should show:
- Soft, visible brush strokes
- Warm, vibrant colour palette
- Atmospheric, painterly rendering

If the style isn't convincing yet, consider increasing `num_epochs` in `config.yaml` and re-running training.

In [ ]:
# ── Test: Impressionism LoRA ───────────────────────────────────────────────────
# Load the best saved Impressionism LoRA and generate sample images.
# Images should show soft brush strokes, warm colours, and impressionistic atmosphere.

save_dir_a  = SAVE_DIR / STYLE_A_NAME.lower()
lora_path_a = save_dir_a / "best"
if not lora_path_a.exists():
    lora_path_a = save_dir_a / "final"
assert lora_path_a.exists(), f"No saved weights found at {lora_path_a} — run training first."

print(f"Loading {STYLE_A_NAME} LoRA from {lora_path_a} …")
pipe_test_a = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, use_safetensors=True, variant="fp16",
).to(device)
pipe_test_a.set_progress_bar_config(disable=True)
pipe_test_a.load_lora_weights(str(lora_path_a), adapter_name="impressionism")
pipe_test_a.set_adapters(["impressionism"], adapter_weights=[1.0])
print(f"✓ {STYLE_A_NAME} LoRA loaded")

print(f'\nGenerating {N_TEST_IMAGES} test image(s) — prompt: "{TEST_PROMPT}"')
images = []
for i in range(N_TEST_IMAGES):
    generator = torch.Generator(device=device).manual_seed(SEED + i)
    with torch.no_grad():
        img = pipe_test_a(
            prompt              = TEST_PROMPT,
            negative_prompt     = NEGATIVE_PROMPT,
            num_inference_steps = N_INFERENCE_STEPS,
            guidance_scale      = GUIDANCE_SCALE,
            height=IMAGE_SIZE, width=IMAGE_SIZE,
            generator=generator,
        ).images[0]
    images.append(img)

ncols = min(N_TEST_IMAGES, 4)
nrows = math.ceil(N_TEST_IMAGES / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 5))
axes_flat  = np.array(axes).reshape(-1) if N_TEST_IMAGES > 1 else [axes]
for ax, img in zip(axes_flat, images):
    ax.imshow(img); ax.axis("off")
for ax in axes_flat[len(images):]:
    ax.axis("off")
fig.suptitle(f"{STYLE_A_NAME} LoRA — style check\n\"{TEST_PROMPT}\"", fontsize=11)
plt.tight_layout()
plt.show()

del pipe_test_a
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Memory cleared — ready for Ukiyo-e training.")

---
## Part 3b: Ukiyo-e LoRA Fine-Tuning

Now we train a **second, independent LoRA** for the Ukiyo-e style.

- The UNet's LoRA adapter is reset to fresh random weights — the Impressionism weights are already saved.
- The frozen base SDXL weights remain unchanged throughout.
- Ukiyo-e is chosen for **maximum visual contrast** with Impressionism:

| Impressionism | Ukiyo-e |
|---|---|
| Soft gradients | Flat areas of colour |
| Visible brush strokes | Bold outlines |
| Atmospheric perspective | Decorative composition |

This contrast will make the style-mixing payoff in Part 4 as striking as possible.

> After training, we test this LoRA individually before blending both styles in Part 4.

In [ ]:
SKIP_TRAINING_B = False   # ← Set True to skip Ukiyo-e training

# ─────────────────────────────────────────────────────────────────────────────
if not SKIP_TRAINING_B:
    # Reset the LoRA adapter — Impressionism weights are saved; start fresh.
    unet.disable_adapters()
    unet.delete_adapter("default")
    unet.add_adapter(lora_config)
    unet.requires_grad_(False)
    for name, param in unet.named_parameters():
        if "lora" in name.lower():
            param.requires_grad_(True)
    print("LoRA adapter reset ✓  (fresh random weights for Ukiyo-e)\n")

    print(f"Training {STYLE_B_NAME} LoRA for {TRAIN_EPOCHS} epoch(s) …")
    print(f"  Batch size    : {BATCH_SIZE}  (grad accum steps: {GRAD_ACCUM})")
    print(f"  Learning rate : {LEARNING_RATE}")
    print(f"  Max samples   : {MAX_SAMPLES}")

    loader_b = DataLoader(
        WikiArtStyleDataset(train_ds, STYLE_B_ID, STYLE_B_NAME,
                            image_size=IMAGE_SIZE, max_samples=MAX_SAMPLES),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True,
    )

    optimizer_b = torch.optim.AdamW(
        [p for p in unet.parameters() if p.requires_grad],
        lr=LEARNING_RATE, weight_decay=1e-2,
    )

    save_dir_b = SAVE_DIR / STYLE_B_NAME.lower()
    save_dir_b.mkdir(parents=True, exist_ok=True)

    loss_history_b = []
    best_loss_b    = float("inf")

    for epoch in range(1, TRAIN_EPOCHS + 1):
        avg_loss = train_epoch(
            unet, loader_b, optimizer_b, noise_scheduler, vae,
            tokenizer_1, tokenizer_2, text_encoder_1, text_encoder_2,
            device, DTYPE, epoch, STYLE_B_NAME, image_size=IMAGE_SIZE,
        )
        loss_history_b.append(avg_loss)
        print(f"  Epoch {epoch}/{TRAIN_EPOCHS}  loss = {avg_loss:.4f}")

        # Periodic checkpoint
        if epoch % SAVE_EVERY == 0 or epoch == TRAIN_EPOCHS:
            ckpt_path = save_dir_b / f"epoch_{epoch}"
            unet.save_pretrained(str(ckpt_path))
            print(f"    → Checkpoint saved: epoch_{epoch}/")

        # Best checkpoint
        if avg_loss < best_loss_b:
            best_loss_b = avg_loss
            unet.save_pretrained(str(save_dir_b / "best"))
            print(f"    ✓ New best! Loss: {best_loss_b:.4f}  → best/")

    # Final save
    unet.save_pretrained(str(save_dir_b / "final"))
    size_mb = sum(f.stat().st_size for f in (save_dir_b / "final").glob("*")) / 1e6
    print(f"\n✓ {STYLE_B_NAME} LoRA complete — best loss: {best_loss_b:.4f}")
    print(f"  Saved at : {save_dir_b}/  (best/, final/, epoch_N/)")
    print(f"  Size     : {size_mb:.1f} MB (final checkpoint)")

    plot_loss_curve(loss_history_b, STYLE_B_NAME, color="darkorange")
else:
    print(f"SKIP_TRAINING_B = True — skipping {STYLE_B_NAME} training.")
    print(f"  Expected weights at: {SAVE_DIR / STYLE_B_NAME.lower()}/")

### Testing the Ukiyo-e LoRA

We verify the Ukiyo-e LoRA with the same test prompt used for Impressionism.
The output should be **visually distinct**:

- Flat areas of colour (no soft gradients)
- Bold outlines and crisp edges
- Decorative, graphic composition

With both LoRAs individually verified, we are ready to **blend them** in Part 4.

In [ ]:
# ── Test: Ukiyo-e LoRA ────────────────────────────────────────────────────────
# Load the best saved Ukiyo-e LoRA and generate sample images.
# Images should be visually distinct from Impressionism: flat colour, bold outlines.

save_dir_b  = SAVE_DIR / STYLE_B_NAME.lower()
lora_path_b = save_dir_b / "best"
if not lora_path_b.exists():
    lora_path_b = save_dir_b / "final"
assert lora_path_b.exists(), f"No saved weights found at {lora_path_b} — run training first."

print(f"Loading {STYLE_B_NAME} LoRA from {lora_path_b} …")
pipe_test_b = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, use_safetensors=True, variant="fp16",
).to(device)
pipe_test_b.set_progress_bar_config(disable=True)
pipe_test_b.load_lora_weights(str(lora_path_b), adapter_name="ukiyo_e")
pipe_test_b.set_adapters(["ukiyo_e"], adapter_weights=[1.0])
print(f"✓ {STYLE_B_NAME} LoRA loaded")

print(f'\nGenerating {N_TEST_IMAGES} test image(s) — prompt: "{TEST_PROMPT}"')
images = []
for i in range(N_TEST_IMAGES):
    generator = torch.Generator(device=device).manual_seed(SEED + i)
    with torch.no_grad():
        img = pipe_test_b(
            prompt              = TEST_PROMPT,
            negative_prompt     = NEGATIVE_PROMPT,
            num_inference_steps = N_INFERENCE_STEPS,
            guidance_scale      = GUIDANCE_SCALE,
            height=IMAGE_SIZE, width=IMAGE_SIZE,
            generator=generator,
        ).images[0]
    images.append(img)

ncols = min(N_TEST_IMAGES, 4)
nrows = math.ceil(N_TEST_IMAGES / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5, nrows * 5))
axes_flat  = np.array(axes).reshape(-1) if N_TEST_IMAGES > 1 else [axes]
for ax, img in zip(axes_flat, images):
    ax.imshow(img); ax.axis("off")
for ax in axes_flat[len(images):]:
    ax.axis("off")
fig.suptitle(f"{STYLE_B_NAME} LoRA — style check\n\"{TEST_PROMPT}\"", fontsize=11)
plt.tight_layout()
plt.show()

del pipe_test_b
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Memory cleared — both LoRAs verified, ready for style mixing (Part 4).")

---
## Part 4: Style Mixing — The Payoff

### The slider is back

In Part 1, you controlled gender with a single weight:

```python
ctx_mixed = torch.tensor([female_weight, 1 - female_weight])
```

At `female_weight = 1.0` → pure female.  At `0.0` → pure male.  At `0.5` → ambiguous blend.

Today, we do the exact same thing with art movements:

```python
pipe.set_adapters(["impressionism", "ukiyo_e"], adapter_weights=[w_imp, w_ukiyo])
```

At `[1.0, 0.0]` → pure Impressionism.
At `[0.0, 1.0]` → pure Ukiyo-e.
At `[0.5, 0.5]` → something genuinely surprising that neither style alone could produce.

### Why the slider, not just text?

We use **both**:
- The **text prompt** describes the **content** (subject matter, composition).
- The **LoRA weight slider** controls the **style** (learned visual language).

This separation is clean: the text does not know about Impressionism or Ukiyo-e —
only the LoRA adapter carries that knowledge.
The slider controls how much of each adapter's knowledge is activated.

In [ ]:
# ── Free training memory ──────────────────────────────────────────────────────
try:
    del unet, optimizer_a, optimizer_b, loader_a, loader_b
    del text_encoder_1, text_encoder_2, vae
except NameError:
    pass
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ── Load the inference pipeline ───────────────────────────────────────────────
print("Loading SDXL inference pipeline …")
pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    use_safetensors=True,
    variant="fp16",
).to(device)
pipe.set_progress_bar_config(disable=True)
print("✓ Pipeline loaded")

# ── Load both LoRA adapters (best checkpoint; fall back to final) ──────────────
def _best_lora_path(style_name):
    """Return path to best checkpoint, falling back to final or flat save."""
    base = SAVE_DIR / style_name.lower()
    for sub in ("best", "final"):
        p = base / sub
        if p.exists():
            return str(p)
    return str(base)   # legacy flat save (older runs)

lora_path_a = _best_lora_path(STYLE_A_NAME)
lora_path_b = _best_lora_path(STYLE_B_NAME)

pipe.load_lora_weights(lora_path_a, adapter_name="impressionism")
print(f"✓ {STYLE_A_NAME} LoRA loaded  ({lora_path_a})")

pipe.load_lora_weights(lora_path_b, adapter_name="ukiyo_e")
print(f"✓ {STYLE_B_NAME} LoRA loaded  ({lora_path_b})")

print("\nBoth adapters ready — you can now blend them with any weights.")

In [ ]:
# ── Style sweep: Impressionism → Ukiyo-e ─────────────────────────────────────
# Five images, same prompt (from config.yaml), varying adapter weights.

plot_style_sweep(
    pipe, device,
    prompt          = SWEEP_PROMPT,
    negative_prompt = NEGATIVE_PROMPT,
    n_steps         = N_INFERENCE_STEPS,
    guidance_scale  = GUIDANCE_SCALE,
    seed            = SEED,
    image_size      = IMAGE_SIZE,
    save_path       = "style_sweep.png",
)

In [ ]:
# generate_styled() is defined in utils.py.
# impressionism_weight=1.0 → pure Impressionism
# impressionism_weight=0.0 → pure Ukiyo-e
# Any value in between → blended style
# (Direct analogue of female_weight from Part 1)

# ── Example 1: Pure Impressionism ─────────────────────────────────────────────
generate_styled(
    pipe, device,
    prompt               = "a village street in summer, flowers, warm golden light",
    impressionism_weight = 1.0,
    n_images             = N_TEST_IMAGES,
    n_steps              = N_INFERENCE_STEPS,
    guidance_scale       = GUIDANCE_SCALE,
    seed                 = SEED,
    negative_prompt      = NEGATIVE_PROMPT,
    image_size           = IMAGE_SIZE,
)

In [ ]:
# ── Example 2: Pure Ukiyo-e ───────────────────────────────────────────────────
generate_styled(
    pipe, device,
    prompt               = "a village street in summer, flowers, warm golden light",
    impressionism_weight = 0.0,
    n_images             = N_TEST_IMAGES,
    n_steps              = N_INFERENCE_STEPS,
    guidance_scale       = GUIDANCE_SCALE,
    seed                 = SEED,
    negative_prompt      = NEGATIVE_PROMPT,
    image_size           = IMAGE_SIZE,
)

In [ ]:
# ── Example 3: 50 / 50 blend ──────────────────────────────────────────────────
generate_styled(
    pipe, device,
    prompt               = "a village street in summer, flowers, warm golden light",
    impressionism_weight = 0.5,
    n_images             = N_TEST_IMAGES,
    n_steps              = N_INFERENCE_STEPS,
    guidance_scale       = GUIDANCE_SCALE,
    seed                 = SEED,
    negative_prompt      = NEGATIVE_PROMPT,
    image_size           = IMAGE_SIZE,
)

In [ ]:
# ── Example 4: Try your own prompt and weight ─────────────────────────────────
generate_styled(
    pipe, device,
    prompt               = "a lone boat on a calm sea at dusk",
    impressionism_weight = 0.3,    # ← change this!
    n_images             = N_TEST_IMAGES,
    n_steps              = 40,
    guidance_scale       = GUIDANCE_SCALE,
    seed                 = SEED,
    negative_prompt      = NEGATIVE_PROMPT,
    image_size           = IMAGE_SIZE,
)

---
## Summary

### What we built today

| Step | Concept | Connection to Part 1 |
|------|---------|----------------------|
| Loaded SDXL | Diffusers abstracts what you implemented by hand | The 5-line version of your 220-line DDPM |
| Explored WikiArt | Richer conditioning signal than gender labels | Same idea, 27 classes instead of 2 |
| Understood LoRA | Efficient fine-tuning via low-rank matrices | Targets cross-attention — same layers your context embedding used |
| Trained two LoRAs | One per style, ~25 MB each | Analogous to training with class labels |
| Blended styles | `adapter_weights=[w1, w2]` | Exact analogue of `female_weight` slider |

### Key takeaways

1. **LoRA is the dominant fine-tuning paradigm** for diffusion models in production.
   File sizes are shareable. Multiple adapters can be stacked and blended at inference time.

2. **The cross-attention mechanism is where conditioning lives** — in your toy U-Net *and* in SDXL.
   LoRA targets exactly those layers because that is where style information is encoded.

3. **Full fine-tuning is rarely the right choice.**
   For style, concept, or subject adaptation, LoRA achieves comparable quality in a fraction of the compute.

4. **The slider intuition generalises.**
   Any two LoRAs trained on the same base model can be blended with weights.
   This extends to three or more adapters, enabling combinatorial style spaces.

---

## Acknowledgements

- **WikiArt dataset**: [huggan/wikiart](https://huggingface.co/datasets/huggan/wikiart)
- **Stable Diffusion XL**: Podell et al., *SDXL: Improving Latent Diffusion Models for High-Resolution Image Synthesis*, 2023
- **LoRA**: Hu et al., *LoRA: Low-Rank Adaptation of Large Language Models*, 2021
- **Diffusers library**: Hugging Face — [github.com/huggingface/diffusers](https://github.com/huggingface/diffusers)
- **PEFT library**: Hugging Face — [github.com/huggingface/peft](https://github.com/huggingface/peft)

*Tutorial by IndabaX Uganda 2026.*